# PlantMetWiki — Data transformation pipeline overview

This notebook generates three publication-ready artefacts:

1. **Pipeline transformation table** — one row per pipeline stage, with tool name, output artefact, key statistics, and notes. Suitable as Table 1 or supplementary material.

2. **Pipeline figure** — a Graphviz diagram (SVG + PDF + DOT source) showing all stages from PlantCyc input to SPARQL access, including VoID metadata creation and the `graph/void` named graph. The SVG is fully editable in Inkscape or Illustrator.

3. **Named-graph metadata table** (`named_graphs_metadata.csv`) — one row per named graph loaded into Virtuoso, with its triple count, source release file, repository, VoID dataset title/version, Zenodo deposit DOI, and licence/provenance notes. Generated by `scripts/export_named_graph_metadata.py` (reads the VoID Turtle files + release bundles directly; can also regenerate from a live SPARQL endpoint with `--mode endpoint`).

## How to run

```bash
conda activate plantmetwiki-rdf
jupyter notebook notebooks/pipeline_overview.ipynb
```

No running Virtuoso instance required — all numbers are embedded in `PIPELINE_STATS` (Cell 1) and updated once per release.

## Updating numbers after a new release

Edit the `PIPELINE_STATS` dictionary in **Cell 1** with new triple counts from the fresh RDF conversion and Virtuoso load.


In [1]:
import pandas as pd
import graphviz
from pathlib import Path

FIG_DIR = Path('figures/output/figures')
OUT_DIR = Path('figures/output')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# === Pipeline statistics -- update this dict after each new RDF conversion ===
# Based on PlantCyc 17.0.0 -> GPML 2021 (v3) -> RDF -> Virtuoso 7.2
#
# Triple counts below (triples_*) are verified directly against the current
# release bundles with rdflib (NOT trusted from a live Virtuoso instance,
# which may have stale/duplicate graphs loaded from earlier dev/test runs --
# see notebooks/figures/output/named_graphs_metadata.csv, generated by
# scripts/export_named_graph_metadata.py, for the full provenance/DOI table).
PIPELINE_STATS = {
    # -- PlantCyc source --------------------------------------------------
    'plantcyc_pathways'         : 1_162,
    'plantcyc_reactions'        : 1_316,
    'plantcyc_gpml_total'       : 2_478,
    'ncbi_taxa_total'           : 439,
    'ncbi_taxa_in_gpml'         : 425,
    'ncbi_taxa_only_complex'    : 12,
    'ncbi_taxa_absent'          : 6,     # taxa in GPML absent from NCBITaxon OBO Foundry release
    'ncbi_taxa_affected_nodes'  : 37,    # DataNode instances carrying an absent taxon
    'ncbi_taxa_annotated_total' : 18_762,# total annotated DataNodes (for %-absent calc)
    'plantcyc_orgids'           : 455,
    # -- GPML DataNodes ----------------------------------------------------
    'gpml_geneproduct'          : 8_259,
    'gpml_protein'               : 10_715,
    'gpml_metabolite'           : 23_449,
    'gpml_complex_group'        : 1_237,
    'gpml_interactions'         : 34_273,
    'gpml_citations'            : 20_677,
    'geneproduct_annot_pct'     : 98.7,
    'protein_annot_pct'         : 98.8,
    # -- Input validation ---------------------------------------------------
    'validation_errors'         : 5,     # cross-species products, skipped
    'validation_warnings'       : 2,     # protein-multi-species + ORG-code
    # -- RDF validation (validate_rdf.py) ------------------------------------
    'rdf_validation_errors'     : 0,
    # -- RDF triple counts per named graph -----------------------------------
    # Verified 2026-06-25 against the current v3 release bundles (rdflib
    # parse, exact triple count) -- see named_graphs_metadata.csv for the
    # per-graph source file and DOI used for each.
    'triples_pathways'          : 3_826_567,
    'triples_taxextra'          : 30_178,
    'triples_propextra'         : 2_617_839,
    'triples_bgc_plantismash'   : 6_030,
    'triples_bgc_mibig'         : 1_813,
    'triples_ncbitaxon'         : 17_707,   # MIREOT subset: 424 taxa + ancestors
    'triples_void'              : 151,      # merged core VoID (82) + BGC VoID (72), deduplicated
    # -- NCBITaxon ROBOT MIREOT subset --------------------------------------
    'robot_taxa_seed'           : 424,
    'robot_subset_mb'           : 1.4,
    'ncbitaxon_full_mb'         : 1_800,    # full release for comparison
    'ncbitaxon_version'         : '2026-05-13',
    # -- BGC (map-to-rdf) -----------------------------------------------------
    # plantiSMASH v2 pre-calculated database, scoped to Arabidopsis thaliana
    # only (tomato clusters predicted by plantiSMASH use a different genome
    # annotation that doesn't join to PlantCyc gene IDs -- see map-to-rdf
    # README "Scope: Arabidopsis thaliana only").
    'bgc_plantismash_version'   : 'v2',
    'bgc_plantismash_bgcs'      : 65,
    'bgc_plantismash_species'   : 'Arabidopsis thaliana',
    'bgc_mibig_version'         : '4.0',
    'bgc_mibig_bgcs'            : 43,
    'bgc_mibig_bgcs_typed'      : 9,        # typed Arabidopsis thaliana (wp:organism)
    'bgc_mibig_bgcs_untyped'    : 34,       # cluster + genes kept, no species claim
    'bgc_crosslinks'            : 199,      # SPARQL-join-derived, not stored as RDF
    # -- Metabolites ----------------------------------------------------------
    'metabolites_unique'        : 4_577,
    'metabolites_with_inchikey' : 4_111,
}

# Derived totals
PIPELINE_STATS['triples_pathway_graphs'] = (
    PIPELINE_STATS['triples_pathways'] +
    PIPELINE_STATS['triples_taxextra'] +
    PIPELINE_STATS['triples_propextra']
)
PIPELINE_STATS['bgc_triples_total'] = (
    PIPELINE_STATS['triples_bgc_plantismash'] +
    PIPELINE_STATS['triples_bgc_mibig']
)
PIPELINE_STATS['triples_total'] = (
    PIPELINE_STATS['triples_pathway_graphs'] +
    PIPELINE_STATS['bgc_triples_total'] +
    PIPELINE_STATS['triples_ncbitaxon'] +
    PIPELINE_STATS['triples_void']
)
PIPELINE_STATS['n_named_graphs'] = 7  # pathways, taxextra, propextra, bgc-plantismash, bgc-mibig, ncbitaxon, void

print(f"Total Virtuoso triples:    {PIPELINE_STATS['triples_total']:,}  across {PIPELINE_STATS['n_named_graphs']} named graphs")
print(f"Pathway-relevant triples:  {PIPELINE_STATS['triples_pathway_graphs'] + PIPELINE_STATS['bgc_triples_total']:,}")
print(f"BGC triples:               plantiSMASH {PIPELINE_STATS['triples_bgc_plantismash']:,} ({PIPELINE_STATS['bgc_plantismash_bgcs']} BGCs) + "
      f"MIBiG {PIPELINE_STATS['triples_bgc_mibig']:,} ({PIPELINE_STATS['bgc_mibig_bgcs']} BGCs) = {PIPELINE_STATS['bgc_triples_total']:,}")
print(f"VoID metadata graph:       {PIPELINE_STATS['triples_void']:,} triples (describes the 5 content datasets; NCBITaxon VoID not yet loaded live)")
print(f"NCBITaxon MIREOT subset:   {PIPELINE_STATS['triples_ncbitaxon']:,} triples  ({PIPELINE_STATS['robot_subset_mb']} MB  <-  {PIPELINE_STATS['ncbitaxon_full_mb']:,} MB full release)")
print(f"Taxa absent from OBO:      {PIPELINE_STATS['ncbi_taxa_absent']} taxa - {PIPELINE_STATS['ncbi_taxa_affected_nodes']} DataNodes affected ({100*PIPELINE_STATS['ncbi_taxa_affected_nodes']/PIPELINE_STATS['ncbi_taxa_annotated_total']:.1f}%)")


Total Virtuoso triples:    6,500,285  across 7 named graphs
Pathway-relevant triples:  6,482,427
BGC triples:               plantiSMASH 6,030 (65 BGCs) + MIBiG 1,813 (43 BGCs) = 7,843
VoID metadata graph:       151 triples (describes the 5 content datasets; NCBITaxon VoID not yet loaded live)
NCBITaxon MIREOT subset:   17,707 triples  (1.4 MB  <-  1,800 MB full release)
Taxa absent from OBO:      6 taxa - 37 DataNodes affected (0.2%)


---
## Pipeline transformation table

In [2]:
S = PIPELINE_STATS

rows = [
    {
        'Stage': '1. PlantCyc 17.0 (source)',
        'Tool / Script': 'BioCyc flat files (.dat)',
        'Output artefact': f"{S['plantcyc_orgids']} ORG-IDs -> {S['ncbi_taxa_total']} NCBI taxa",
        'Key statistics': f"{S['plantcyc_pathways']:,} pathways - {S['plantcyc_reactions']:,} reactions",
        'Notes': 'PMN/PlantCyc licence',
    },
    {
        'Stage': '2. Input validation',
        'Tool / Script': 'validate_plantcyc_input.py',
        'Output artefact': 'VALIDATION_REPORT.txt, VALIDATION_SUMMARY.tsv',
        'Key statistics': f"{S['validation_errors']} ERRORs skipped - {S['validation_warnings']} WARNINGs",
        'Notes': 'Deterministic build; documented in Table S3',
    },
    {
        'Stage': '3. GPML conversion',
        'Tool / Script': 'build_pathways.py + gpml2rdf-4.0.4.jar',
        'Output artefact': f"{S['plantcyc_gpml_total']:,} GPML2021 files",
        'Key statistics': (f"{S['gpml_geneproduct']:,} GeneProduct - {S['gpml_protein']:,} Protein - "
                           f"{S['gpml_metabolite']:,} Metabolite - {S['gpml_interactions']:,} interactions; "
                           f"Taxon coverage: {S['geneproduct_annot_pct']}% genes - {S['protein_annot_pct']}% proteins"),
        'Notes': 'GPML2021 XSD validated (0 errors)',
    },
    {
        'Stage': '4. Core RDF',
        'Tool / Script': 'gpml-to-rdf (Java + Makefile)',
        'Output artefact': 'graph/pathways',
        'Key statistics': f"{S['triples_pathways']:,} triples",
        'Notes': 'WikiPathways wp: vocabulary',
    },
    {
        'Stage': '5. Taxonomy extra RDF',
        'Tool / Script': 'create_gpml_taxonomy_extra_rdf.py',
        'Output artefact': 'graph/gpml-taxonomy-extra',
        'Key statistics': (f"{S['triples_taxextra']:,} triples - "
                           f"{S['ncbi_taxa_in_gpml']}/{S['ncbi_taxa_total']} NCBI taxa annotated - "
                           f"{S['ncbi_taxa_absent']} taxa absent from OBO Foundry "
                           f"({S['ncbi_taxa_affected_nodes']} DataNodes affected)"),
        'Notes': 'wp:organism ncbi:XXXX per DataNode; absent taxa documented in explore_taxonomy_rdf.ipynb',
    },
    {
        'Stage': '6. Properties extra RDF',
        'Tool / Script': 'create_gpml_properties_extra_rdf.py',
        'Output artefact': 'graph/gpml-properties-extra',
        'Key statistics': f"{S['triples_propextra']:,} triples",
        'Notes': 'PlantCyc key-value metadata preserved',
    },
    {
        'Stage': '7. RDF validation',
        'Tool / Script': 'validate_rdf.py',
        'Output artefact': 'RDF_VALIDATION_REPORT.txt',
        'Key statistics': (f"{S['rdf_validation_errors']} errors - "
                           f"{S['triples_pathways']:,} + {S['triples_taxextra']:,} + "
                           f"{S['triples_propextra']:,} triples across 3 graphs verified"),
        'Notes': 'TTL syntax + content checks on all output files before Virtuoso load',
    },
    {
        'Stage': '8. NCBITaxon ontology (ROBOT MIREOT)',
        'Tool / Script': 'load-ncbitaxon.sh --subset plantmetwiki (Snorql-UI)',
        'Output artefact': 'graph/ncbitaxon',
        'Key statistics': (f"{S['triples_ncbitaxon']:,} triples - "
                           f"MIREOT subset: {S['robot_taxa_seed']} seed taxa + ancestors - "
                           f"{S['robot_subset_mb']} MB "
                           f"(from {S['ncbitaxon_full_mb']:,} MB full release, v{S['ncbitaxon_version']})"),
        'Notes': 'ROBOT v1.9.6 extract --method MIREOT; OBO Foundry CC0 licence',
    },
    {
        'Stage': '9. BGC integration',
        'Tool / Script': 'map-to-rdf (convert_bgc_to_rdf.py)',
        'Output artefact': 'graph/bgc-mibig, graph/bgc-plantismash',
        'Key statistics': (f"plantiSMASH {S['bgc_plantismash_version']}: {S['bgc_plantismash_bgcs']} BGCs "
                           f"({S['bgc_plantismash_species']} only) - {S['triples_bgc_plantismash']:,} triples; "
                           f"MIBiG {S['bgc_mibig_version']}: {S['bgc_mibig_bgcs']} BGCs "
                           f"({S['bgc_mibig_bgcs_typed']} typed A. thaliana, {S['bgc_mibig_bgcs_untyped']} untyped) - "
                           f"{S['triples_bgc_mibig']:,} triples - {S['bgc_crosslinks']} pathway crosslinks (SPARQL-join derived)"),
        'Notes': 'BGC-to-pathway linking done at the SPARQL layer, not during RDF generation',
    },
    {
        'Stage': '10. VoID metadata creation',
        'Tool / Script': 'create_void_from_metadata.py (gpml-to-rdf) + create_void_bgc.py (map-to-rdf)',
        'Output artefact': 'void-plantcyc17.0.0-gpml2021-v3.ttl, void-bgc.ttl -> graph/void',
        'Key statistics': (f"{S['triples_void']:,} triples - 5 void:Dataset descriptions "
                           f"(core, taxonomy-extra, properties-extra, plantismash-{S['bgc_plantismash_version']}, "
                           f"mibig-{S['bgc_mibig_version']}) + 1 void:Linkset"),
        'Notes': 'Deposit DOIs, licences, versions, byte sizes; see named_graphs_metadata.csv for full table',
    },
    {
        'Stage': '11. Triplestore (Virtuoso 7.2)',
        'Tool / Script': 'docker compose up -d virtuoso',
        'Output artefact': 'SPARQL endpoint + Snorql-UI browser',
        'Key statistics': f"{S['triples_total']:,} total triples across {S['n_named_graphs']} named graphs",
        'Notes': 'https://sparql-plantmetwiki.bioinformatics.nl',
    },
]

df = pd.DataFrame(rows)

# Save as TSV
tsv_path = OUT_DIR / 'pipeline_table.tsv'
df.to_csv(tsv_path, sep='\t', index=False)
print(f"Saved: {tsv_path}")

# Display styled HTML in notebook
from IPython.display import display, HTML
styled = (df.style
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap',
                       'font-size': '12px', 'padding': '4px 8px'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color','#2c3e50'),
                                      ('color','white'),('font-size','12px'),
                                      ('padding','6px 10px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f8f9fa')]},
    ])
    .hide(axis='index')
)
display(styled)


Saved: figures/output/pipeline_table.tsv


Stage,Tool / Script,Output artefact,Key statistics,Notes
1. PlantCyc 17.0 (source),BioCyc flat files (.dat),455 ORG-IDs -> 439 NCBI taxa,"1,162 pathways - 1,316 reactions",PMN/PlantCyc licence
2. Input validation,validate_plantcyc_input.py,"VALIDATION_REPORT.txt, VALIDATION_SUMMARY.tsv",5 ERRORs skipped - 2 WARNINGs,Deterministic build; documented in Table S3
3. GPML conversion,build_pathways.py + gpml2rdf-4.0.4.jar,"2,478 GPML2021 files","8,259 GeneProduct - 10,715 Protein - 23,449 Metabolite - 34,273 interactions; Taxon coverage: 98.7% genes - 98.8% proteins",GPML2021 XSD validated (0 errors)
4. Core RDF,gpml-to-rdf (Java + Makefile),graph/pathways,"3,826,567 triples",WikiPathways wp: vocabulary
5. Taxonomy extra RDF,create_gpml_taxonomy_extra_rdf.py,graph/gpml-taxonomy-extra,"30,178 triples - 425/439 NCBI taxa annotated - 6 taxa absent from OBO Foundry (37 DataNodes affected)",wp:organism ncbi:XXXX per DataNode; absent taxa documented in explore_taxonomy_rdf.ipynb
6. Properties extra RDF,create_gpml_properties_extra_rdf.py,graph/gpml-properties-extra,"2,617,839 triples",PlantCyc key-value metadata preserved
7. RDF validation,validate_rdf.py,RDF_VALIDATION_REPORT.txt,"0 errors - 3,826,567 + 30,178 + 2,617,839 triples across 3 graphs verified",TTL syntax + content checks on all output files before Virtuoso load
8. NCBITaxon ontology (ROBOT MIREOT),load-ncbitaxon.sh --subset plantmetwiki (Snorql-UI),graph/ncbitaxon,"17,707 triples - MIREOT subset: 424 seed taxa + ancestors - 1.4 MB (from 1,800 MB full release, v2026-05-13)",ROBOT v1.9.6 extract --method MIREOT; OBO Foundry CC0 licence
9. BGC integration,map-to-rdf (convert_bgc_to_rdf.py),"graph/bgc-mibig, graph/bgc-plantismash","plantiSMASH v2: 65 BGCs (Arabidopsis thaliana only) - 6,030 triples; MIBiG 4.0: 43 BGCs (9 typed A. thaliana, 34 untyped) - 1,813 triples - 199 pathway crosslinks (SPARQL-join derived)","BGC-to-pathway linking done at the SPARQL layer, not during RDF generation"
10. VoID metadata creation,create_void_from_metadata.py (gpml-to-rdf) + create_void_bgc.py (map-to-rdf),"void-plantcyc17.0.0-gpml2021-v3.ttl, void-bgc.ttl -> graph/void","151 triples - 5 void:Dataset descriptions (core, taxonomy-extra, properties-extra, plantismash-v2, mibig-4.0) + 1 void:Linkset","Deposit DOIs, licences, versions, byte sizes; see named_graphs_metadata.csv for full table"


---
## Named-graph metadata table

One row per named graph loaded into Virtuoso: triple count, VoID dataset title/version, Zenodo deposit DOI, source release file, and provenance notes. Generated by `scripts/export_named_graph_metadata.py` from the actual VoID Turtle files and release bundles (not from a live Virtuoso instance, to avoid picking up stale/duplicate dev-only graphs).

In [3]:
import subprocess

subprocess.run(['python', '../scripts/export_named_graph_metadata.py'], check=True, cwd='.')

named_graphs_df = pd.read_csv(OUT_DIR / 'named_graphs_metadata.csv')

from IPython.display import display
display(named_graphs_df.style
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap',
                       'font-size': '11px', 'padding': '4px 8px'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color','#27ae60'),
                                      ('color','white'),('font-size','11px'),
                                      ('padding','6px 10px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f8f9fa')]},
    ])
    .hide(axis='index')
)

Saved: /Users/elenadelpup/Library/CloudStorage/OneDrive-WageningenUniversity&Research/PhD/software/plantwiki/gpml-to-rdf/notebooks/figures/output/named_graphs_metadata.csv  (8 rows)
                                                          named_graph  triples                                                                                                                dataset_title                    version    created                             deposit_doi                                                                                                           source_uri                                                               source_file                     repo                                                                                                                                                                                                                                                                notes
             http://rdf-plantmetwiki.bioinformatics.nl/grap

named_graph,triples,dataset_title,version,created,deposit_doi,source_uri,source_file,repo,notes
http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways,3826567,PlantMetWiki core RDF derived from PlantCyc to WikiPathways: BioCyc flat-file to GPML2021 conversion pipeline,plantcyc17.0.0-gpml2021-v3,2026-06-11,https://doi.org/10.5281/zenodo.18404067,https://doi.org/10.5281/zenodo.20560642,all-plantcyc17.0.0-gpml2021-v3.ttl,gpml-to-rdf,"Core WikiPathways RDF (pathways + reactions, both typed wp:Pathway)."
http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra,30178,PlantMetWiki taxonomy extra RDF derived from PlantCyc to WikiPathways: BioCyc flat-file to GPML2021 conversion pipeline,plantcyc17.0.0-gpml2021-v3,2026-06-11,https://doi.org/10.5281/zenodo.18404067,https://doi.org/10.5281/zenodo.20560642,all_gpml_taxonomy_extra-plantcyc17.0.0-gpml2021-v3.ttl,gpml-to-rdf,Per-DataNode wp:organism species annotations + Viridiplantae root.
http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-properties-extra,2617839,PlantMetWiki GPML property extra RDF derived from PlantCyc to WikiPathways: BioCyc flat-file to GPML2021 conversion pipeline,plantcyc17.0.0-gpml2021-v3,2026-06-11,https://doi.org/10.5281/zenodo.18404067,https://doi.org/10.5281/zenodo.20560642,all_gpml_properties_extra-plantcyc17.0.0-gpml2021-v3.ttl,gpml-to-rdf,PlantCyc/GPML key-value Property elements preserved as pmw:gpmlProperty.
http://rdf-plantmetwiki.bioinformatics.nl/graph/bgc-plantismash,6030,PlantMetWiki BGC crosslinks — plantiSMASH v2,plantismash-v2,2026-06-11,https://doi.org/10.5281/zenodo.20345133,https://raw.githubusercontent.com/plantismash/plantismash-database/main/data/plantismash_v2_clusters_minimal.json,plantismash.ttl,map-to-rdf,"plantiSMASH v2 pre-calculated database, 65 BGCs, Arabidopsis thaliana only."
http://rdf-plantmetwiki.bioinformatics.nl/graph/bgc-mibig,1813,PlantMetWiki BGC crosslinks — MIBiG 4.0,mibig-4.0,2026-06-11,https://doi.org/10.5281/zenodo.20345133,https://github.com/plantismash/plantismash/blob/master/antismash/generic_modules/knownclusterblast/knownclusters.txt,mibig.ttl,map-to-rdf,"MIBiG 4.0, 43 BGCs (9 typed Arabidopsis thaliana, 34 untyped/no species)."
http://rdf-plantmetwiki.bioinformatics.nl/graph/ncbitaxon,17707,nan,nan,nan,nan,nan,"(ROBOT MIREOT extract from ncbitaxon.owl, not bundled as a repo file)",Snorql-UI,"MIREOT subset: 424 seed taxa + ancestors, v2026-05-13, 1.4 MB. VoID description exists only as a static file (db/data/void-ncbitaxon.ttl) -- not yet loaded into the live `void` named graph, so it won't appear in a live VoIDHeader.rq run."
http://rdf-plantmetwiki.bioinformatics.nl/void,151,nan,nan,nan,nan,nan,"void-plantcyc17.0.0-gpml2021-v3.ttl + void-bgc.ttl (merged, deduplicated)",gpml-to-rdf + map-to-rdf,"VoID metadata graph itself: describes the 5 content datasets above (core, taxonomy-extra, properties-extra, plantismash-v2, mibig-4.0). 82 triples from the core VoID + 72 from the BGC VoID, with ~3 duplicate publisher/organization triples deduplicated on load."
TOTAL,6500285,nan,nan,nan,nan,nan,nan,nan,7 named graphs


---
## Pipeline figure (Graphviz SVG)

In [4]:
S = PIPELINE_STATS

dot = graphviz.Digraph(
    name='PlantMetWiki_pipeline',
    comment='PlantMetWiki data transformation pipeline',
    format='svg',
    engine='dot',
)
dot.attr(rankdir='TB', nodesep='0.5', ranksep='0.7',
         fontname='Arial', fontsize='11', bgcolor='white')
dot.attr('node', fontname='Arial', fontsize='10', margin='0.15,0.10')
dot.attr('edge', fontname='Arial', fontsize='9', color='#555555')

C_SRC   = '#dbe9f4'
C_TOOL  = '#fef3cd'
C_GRAPH = '#d5f5e3'
C_ACC   = '#e8daef'

# -- Source data --------------------------------------------------------------
with dot.subgraph(name='cluster_source') as c:
    c.attr(label='Source data', style='filled,rounded', fillcolor=C_SRC,
           color='#2980b9', penwidth='1.5', fontcolor='#2980b9', fontsize='11', fontname='Arial')
    c.node('plantcyc', shape='cylinder', style='filled', fillcolor='white',
           label=(f'PlantCyc 17.0\n'
                  f'{S["plantcyc_pathways"]:,} pathways - {S["plantcyc_reactions"]:,} reactions\n'
                  f'{S["plantcyc_orgids"]} ORG-IDs -> {S["ncbi_taxa_total"]} NCBI taxa'))
    c.node('mibig', shape='cylinder', style='filled', fillcolor='white',
           label=(f'MIBiG {S["bgc_mibig_version"]}\n'
                  f'{S["bgc_mibig_bgcs"]} BGCs ({S["bgc_mibig_bgcs_typed"]} A. thaliana, '
                  f'{S["bgc_mibig_bgcs_untyped"]} untyped)'))
    c.node('plantismash', shape='cylinder', style='filled', fillcolor='white',
           label=(f'plantiSMASH {S["bgc_plantismash_version"]}\n'
                  f'{S["bgc_plantismash_bgcs"]} predicted BGCs\n'
                  f'{S["bgc_plantismash_species"]} only'))
    c.node('ncbitaxon_src', shape='cylinder', style='filled', fillcolor='white',
           label='NCBITaxon (OBO Foundry)\nCC0 licence')
    c.node('wikidata_src', shape='cylinder', style='filled', fillcolor='white',
           label='Wikidata\n(federated via InChIKey)')

# -- Pipeline tools ------------------------------------------------------------
with dot.subgraph(name='cluster_tools') as c:
    c.attr(label='Pipeline tools', style='filled,rounded', fillcolor=C_TOOL,
           color='#e67e22', penwidth='1.5', fontcolor='#e67e22', fontsize='11', fontname='Arial')
    c.node('validate', shape='diamond', style='filled', fillcolor='white',
           label=(f'validate_plantcyc_input.py\n'
                  f'{S["validation_errors"]} ERRORs (skipped) - {S["validation_warnings"]} WARNINGs'))
    c.node('cyc2wiki', shape='box', style='filled,rounded', fillcolor='white',
           label=(f'Cyc_to_wiki + build_pathways.py\n'
                  f'{S["plantcyc_gpml_total"]:,} GPML2021 files\n'
                  f'{S["gpml_geneproduct"]:,} GeneProduct - {S["gpml_protein"]:,} Protein\n'
                  f'{S["gpml_metabolite"]:,} Metabolite - {S["gpml_interactions"]:,} interactions'))
    c.node('xsd_val', shape='box', style='filled,rounded', fillcolor='white',
           label='test_gpml_files.py\nGPML2021 XSD validation\n0 errors')
    c.node('gpml2rdf', shape='box', style='filled,rounded', fillcolor='white',
           label=('gpml-to-rdf  (Java + Python scripts)\n'
                  'gpml2rdf-4.0.4.jar - taxonomy extra - properties extra\n'
                  f'Core {S["triples_pathways"]:,} + Tax {S["triples_taxextra"]:,} + Prop {S["triples_propextra"]:,} triples'))
    c.node('rdf_validate', shape='diamond', style='filled', fillcolor='white', ordering='out',
           label=(f'validate_rdf.py\n'
                  f'{S["rdf_validation_errors"]} errors - {S["triples_pathway_graphs"]:,} triples verified'))
    c.node('load_ncbi', shape='box', style='filled,rounded', fillcolor='white', ordering='out',
           label=(f'load-ncbitaxon.sh --subset plantmetwiki\n'
                  f'ROBOT v1.9.6  extract --method MIREOT\n'
                  f'{S["robot_taxa_seed"]} seed taxa + ancestors  ->  {S["triples_ncbitaxon"]:,} triples\n'
                  f'{S["robot_subset_mb"]} MB  (full release: {S["ncbitaxon_full_mb"]:,} MB)'))
    c.node('map2rdf', shape='box', style='filled,rounded', fillcolor='white', ordering='out',
           label=(f'map-to-rdf  (convert_bgc_to_rdf.py)\n'
                  f'Scope: {S["bgc_plantismash_species"]} only\n'
                  f'{S["bgc_crosslinks"]} BGC-pathway crosslinks (SPARQL join) - '
                  f'{S["triples_bgc_plantismash"]+S["triples_bgc_mibig"]:,} triples'))
    c.node('void_create', shape='box', style='filled,rounded', fillcolor='white', ordering='out',
           label=(f'create_void_from_metadata.py (core)\n'
                  f'create_void_bgc.py (BGC)\n'
                  f'5 void:Dataset + 1 void:Linkset  ->  {S["triples_void"]:,} triples'))

# -- Virtuoso named graphs ------------------------------------------------------
# graph/pathways declared first, immediately followed by the other two
# "core" RDF graphs (taxonomy-extra, properties-extra) it's most closely
# related to, so dot's layout keeps them adjacent.
with dot.subgraph(name='cluster_virtuoso') as c:
    c.attr(label=f'Virtuoso 7.2  --  {S["triples_total"]:,} total triples  ({S["n_named_graphs"]} named graphs)',
           style='filled,rounded', fillcolor=C_GRAPH,
           color='#27ae60', penwidth='1.5', fontcolor='#27ae60', fontsize='11', fontname='Arial')
    # graph/pathways, taxonomy-extra and properties-extra are all derived
    # from the same gpml-to-rdf conversion run -- group them in their own
    # nested cluster (dashed border) so they stay visually adjacent.
    # Clusters are a hard grouping constraint in dot, unlike same-rank +
    # invisible-edge ordering, which dot's crossing-minimisation can (and
    # did) still reorder/scatter once other edges entered the same rank.
    with c.subgraph(name='cluster_core_graphs') as core:
        core.attr(label='core RDF (gpml-to-rdf)', style='dashed', color='#27ae60',
                   fontsize='9', fontname='Arial', fontcolor='#27ae60')
        core.node('g_pathways', shape='tab', style='filled', fillcolor='white',
                   label=f'graph/pathways\n{S["triples_pathways"]:,} triples')
        core.node('g_taxextra', shape='tab', style='filled', fillcolor='white',
                   label=(f'graph/gpml-taxonomy-extra\n{S["triples_taxextra"]:,} triples\n'
                          f'{S["ncbi_taxa_in_gpml"]}/{S["ncbi_taxa_total"]} taxa - '
                          f'{S["ncbi_taxa_absent"]} absent from OBO Foundry'))
        core.node('g_propextra', shape='tab', style='filled', fillcolor='white',
                   label=f'graph/gpml-properties-extra\n{S["triples_propextra"]:,} triples')
    c.node('g_bgc_plantismash', shape='tab', style='filled', fillcolor='white',
           label=f'graph/bgc-plantismash\n{S["triples_bgc_plantismash"]:,} triples')
    c.node('g_bgc_mibig', shape='tab', style='filled', fillcolor='white',
           label=f'graph/bgc-mibig\n{S["triples_bgc_mibig"]:,} triples')
    c.node('g_ncbi', shape='tab', style='filled', fillcolor='white',
           label=(f'graph/ncbitaxon\n{S["triples_ncbitaxon"]:,} triples\n'
                  f'MIREOT subset  v{S["ncbitaxon_version"]}'))
    c.node('g_void', shape='tab', style='filled', fillcolor='white',
           label=(f'graph/void\n{S["triples_void"]:,} triples\n'
                  f'5 void:Dataset descriptions + 1 void:Linkset'))

# -- Access & results -----------------------------------------------------------
# sparql lives here (not in cluster_virtuoso) -- it's the access point onto
# the triplestore, not a named graph itself. Declared first so it sits at
# the top of this cluster, directly under the Virtuoso box above.
with dot.subgraph(name='cluster_access') as c:
    c.attr(label='Access & results', style='filled,rounded', fillcolor=C_ACC,
           color='#8e44ad', penwidth='1.5', fontcolor='#8e44ad', fontsize='11', fontname='Arial')
    c.node('sparql', shape='component', style='filled', fillcolor='white',
           label='SPARQL endpoint\nhttps://sparql-plantmetwiki.bioinformatics.nl')
    c.node('snorql', shape='box', style='filled,rounded', fillcolor='white',
           label='Snorql-UI\nhttps://plantmetwiki.bioinformatics.nl')
    c.node('notebooks', shape='note', style='filled', fillcolor='white',
           label=(f'Jupyter notebooks\n'
                  f'{S["metabolites_unique"]:,} metabolites ({S["metabolites_with_inchikey"]:,} with InChIKey)\n'
                  f'Figures - Cross-species inference - Federated queries'))
    c.node('sparql_examples', shape='note', style='filled', fillcolor='white',
           label='SPARQL query examples\ngithub.com/pathway-lod/SPARQLQueries\ncustomizable via Snorql-UI settings')
    c.node('tutorials', shape='note', style='filled', fillcolor='white',
           label='Tutorial pages\nStep-by-step SPARQL - federated queries')
    # sparql centred above snorql + notebooks side by side beneath it
    c.edge('sparql', 'snorql', style='invis')
    c.edge('sparql', 'notebooks', style='invis')

# -- Edges ----------------------------------------------------------------------
dot.edge('plantcyc', 'validate')
dot.edge('validate', 'cyc2wiki', label='5 ERRORs skipped')
dot.edge('cyc2wiki', 'xsd_val', label='2,478 GPML files')
dot.edge('xsd_val', 'gpml2rdf', label='XSD pass')
dot.edge('gpml2rdf', 'rdf_validate', label='TTL output')
dot.edge('rdf_validate', 'g_pathways', label='0 errors')
dot.edge('rdf_validate', 'g_taxextra')
dot.edge('rdf_validate', 'g_propextra')
dot.edge('mibig', 'map2rdf')
dot.edge('plantismash', 'map2rdf')
dot.edge('map2rdf', 'g_bgc_plantismash')
dot.edge('map2rdf', 'g_bgc_mibig')
dot.edge('ncbitaxon_src', 'load_ncbi')
dot.edge('load_ncbi', 'g_ncbi')
dot.edge('rdf_validate', 'void_create', style='dashed')
dot.edge('map2rdf', 'void_create', style='dashed')
dot.edge('void_create', 'g_void')
dot.edge('g_pathways', 'sparql')
dot.edge('g_taxextra', 'sparql')
dot.edge('g_propextra', 'sparql')
dot.edge('g_bgc_plantismash', 'sparql')
dot.edge('g_bgc_mibig', 'sparql')
dot.edge('g_ncbi', 'sparql')
dot.edge('g_void', 'sparql')
dot.edge('sparql', 'snorql')
dot.edge('sparql', 'notebooks')
dot.edge('snorql', 'sparql_examples', style='dashed', label='loads from GitHub')
dot.edge('snorql', 'tutorials', style='dashed')
dot.edge('wikidata_src', 'notebooks', style='dashed', label='federated SPARQL')
dot.edge('wikidata_src', 'sparql_examples', style='dashed', label='federated SPARQL')

# -- Render -----------------------------------------------------------------------
dot_src_path = str(FIG_DIR / 'pipeline_overview')
dot.render(dot_src_path, cleanup=False)
(FIG_DIR / 'pipeline_overview.dot').write_text(dot.source, encoding='utf-8')

print(f"Saved SVG: {dot_src_path}.svg")
print(f"Saved DOT: {dot_src_path}.dot")


Saved SVG: figures/output/figures/pipeline_overview.svg
Saved DOT: figures/output/figures/pipeline_overview.dot


In [5]:
S = PIPELINE_STATS

# -- Horizontal / slide-friendly version (rankdir=LR) -------------------------
dot_h = graphviz.Digraph(
    name='PlantMetWiki_pipeline_horizontal',
    comment='PlantMetWiki pipeline -- horizontal layout for slides',
    format='svg',
    engine='dot',
)
dot_h.attr(rankdir='LR', nodesep='0.35', ranksep='0.6',
           fontname='Arial', fontsize='11', bgcolor='white')
dot_h.attr('node', fontname='Arial', fontsize='9', margin='0.12,0.08')
dot_h.attr('edge', fontname='Arial', fontsize='8', color='#555555')

C_SRC   = '#dbe9f4'
C_TOOL  = '#fef3cd'
C_GRAPH = '#d5f5e3'
C_ACC   = '#e8daef'

# -- Source data --------------------------------------------------------------
with dot_h.subgraph(name='cluster_source') as c:
    c.attr(label='Source data', style='filled,rounded', fillcolor=C_SRC,
           color='#2980b9', penwidth='1.5', fontcolor='#2980b9', fontsize='10', fontname='Arial')
    c.node('plantcyc', shape='cylinder', style='filled', fillcolor='white',
           label=(f'PlantCyc 17.0\n'
                  f'{S["plantcyc_pathways"]:,} pathways\n'
                  f'{S["plantcyc_reactions"]:,} reactions\n'
                  f'{S["ncbi_taxa_total"]} NCBI taxa'))
    c.node('ncbitaxon_src', shape='cylinder', style='filled', fillcolor='white',
           label='NCBITaxon\n(OBO Foundry, CC0)')
    c.node('mibig', shape='cylinder', style='filled', fillcolor='white',
           label=(f'MIBiG {S["bgc_mibig_version"]}\n'
                  f'{S["bgc_mibig_bgcs"]} BGCs ({S["bgc_mibig_bgcs_typed"]} A. thaliana)'))
    c.node('plantismash', shape='cylinder', style='filled', fillcolor='white',
           label=(f'plantiSMASH {S["bgc_plantismash_version"]}\n'
                  f'{S["bgc_plantismash_bgcs"]} BGCs ({S["bgc_plantismash_species"]})'))
    c.node('wikidata_src', shape='cylinder', style='filled', fillcolor='white',
           label='Wikidata\n(federated)')

# -- Pipeline tools ------------------------------------------------------------
with dot_h.subgraph(name='cluster_tools') as c:
    c.attr(label='Pipeline tools', style='filled,rounded', fillcolor=C_TOOL,
           color='#e67e22', penwidth='1.5', fontcolor='#e67e22', fontsize='10', fontname='Arial')
    c.node('validate', shape='diamond', style='filled', fillcolor='white',
           label=(f'validate_plantcyc_input.py\n'
                  f'{S["validation_errors"]} ERRORs - {S["validation_warnings"]} WARNINGs'))
    c.node('cyc2wiki', shape='box', style='filled,rounded', fillcolor='white',
           label=(f'Cyc_to_wiki + build_pathways.py\n'
                  f'{S["plantcyc_gpml_total"]:,} GPML2021 files\n'
                  f'{S["gpml_geneproduct"]:,} GP - {S["gpml_protein"]:,} Prot - '
                  f'{S["gpml_metabolite"]:,} Met'))
    c.node('xsd_val', shape='box', style='filled,rounded', fillcolor='white',
           label='XSD validation\n0 errors')
    c.node('gpml2rdf', shape='box', style='filled,rounded', fillcolor='white',
           label=(f'gpml-to-rdf\n'
                  f'Core {S["triples_pathways"]:,} triples\n'
                  f'+ Tax {S["triples_taxextra"]:,} - Prop {S["triples_propextra"]:,}'))
    c.node('rdf_validate', shape='diamond', style='filled', fillcolor='white', ordering='out',
           label=(f'validate_rdf.py\n'
                  f'0 errors - {S["triples_pathway_graphs"]:,} triples'))
    c.node('load_ncbi', shape='box', style='filled,rounded', fillcolor='white', ordering='out',
           label=(f'ROBOT MIREOT\n'
                  f'{S["robot_taxa_seed"]} taxa -> {S["triples_ncbitaxon"]:,} triples\n'
                  f'{S["robot_subset_mb"]} MB (from {S["ncbitaxon_full_mb"]:,} MB)'))
    c.node('map2rdf', shape='box', style='filled,rounded', fillcolor='white', ordering='out',
           label=(f'map-to-rdf  ({S["bgc_plantismash_species"]})\n'
                  f'{S["bgc_crosslinks"]} BGC crosslinks'))
    c.node('void_create', shape='box', style='filled,rounded', fillcolor='white', ordering='out',
           label=(f'create_void_*.py\n'
                  f'{S["triples_void"]:,} triples'))

# -- Virtuoso named graphs ------------------------------------------------------
with dot_h.subgraph(name='cluster_virtuoso') as c:
    c.attr(label=f'Virtuoso 7.2\n{S["triples_total"]:,} triples ({S["n_named_graphs"]} graphs)',
           style='filled,rounded', fillcolor=C_GRAPH,
           color='#27ae60', penwidth='1.5', fontcolor='#27ae60', fontsize='10', fontname='Arial')
    # graph/pathways, taxonomy-extra and properties-extra are derived
    # from the same gpml-to-rdf run -- group them in a nested cluster
    # (dashed border) so they stay visually adjacent regardless of dot's
    # crossing-minimisation (same-rank + invisible-edge ordering proved
    # unreliable once other edges entered the same rank).
    with c.subgraph(name='cluster_core_graphs') as core:
        core.attr(label='core RDF', style='dashed', color='#27ae60',
                   fontsize='8', fontname='Arial', fontcolor='#27ae60')
        core.node('g_pathways', shape='tab', style='filled', fillcolor='white',
                   label=f'graph/pathways\n{S["triples_pathways"]:,}')
        core.node('g_taxextra', shape='tab', style='filled', fillcolor='white',
                   label=(f'graph/taxonomy-extra\n{S["triples_taxextra"]:,} - '
                          f'{S["ncbi_taxa_absent"]} absent'))
        core.node('g_propextra', shape='tab', style='filled', fillcolor='white',
                   label=f'graph/properties-extra\n{S["triples_propextra"]:,}')
    c.node('g_bgc_plantismash', shape='tab', style='filled', fillcolor='white',
           label=f'graph/bgc-plantismash\n{S["triples_bgc_plantismash"]:,}')
    c.node('g_bgc_mibig', shape='tab', style='filled', fillcolor='white',
           label=f'graph/bgc-mibig\n{S["triples_bgc_mibig"]:,}')
    c.node('g_ncbi', shape='tab', style='filled', fillcolor='white',
           label=(f'graph/ncbitaxon\n{S["triples_ncbitaxon"]:,}\n'
                  f'MIREOT v{S["ncbitaxon_version"]}'))
    c.node('g_void', shape='tab', style='filled', fillcolor='white',
           label=f'graph/void\n{S["triples_void"]:,}')

# -- Access & results ------------------------------------------------------------
# sparql lives here, not in cluster_virtuoso -- it's the access point onto
# the triplestore, not a named graph itself. Declared first so it sits
# closest to the Virtuoso box in the LR layout.
with dot_h.subgraph(name='cluster_access') as c:
    c.attr(label='Access & results', style='filled,rounded', fillcolor=C_ACC,
           color='#8e44ad', penwidth='1.5', fontcolor='#8e44ad', fontsize='10', fontname='Arial')
    c.node('sparql', shape='component', style='filled', fillcolor='white',
           label='SPARQL endpoint')
    c.node('snorql', shape='box', style='filled,rounded', fillcolor='white',
           label='Snorql-UI\nbrowser interface')
    c.edge('sparql', 'snorql', style='invis')
    c.node('sparql_examples', shape='note', style='filled', fillcolor='white',
           label='SPARQL examples\n(GitHub, customizable)')
    c.node('tutorials', shape='note', style='filled', fillcolor='white',
           label='Tutorial pages\nfederated queries')
    c.node('notebooks', shape='note', style='filled', fillcolor='white',
           label=(f'Jupyter notebooks\n'
                  f'{S["metabolites_unique"]:,} metabolites\n'
                  f'figures - inference'))

# -- Edges ------------------------------------------------------------------------
dot_h.edge('plantcyc', 'validate')
dot_h.edge('validate', 'cyc2wiki', label='5 skipped')
dot_h.edge('cyc2wiki', 'xsd_val')
dot_h.edge('xsd_val', 'gpml2rdf', label='XSD pass')
dot_h.edge('gpml2rdf', 'rdf_validate')
dot_h.edge('rdf_validate', 'g_pathways', label='0 errors')
dot_h.edge('rdf_validate', 'g_taxextra')
dot_h.edge('rdf_validate', 'g_propextra')
dot_h.edge('mibig', 'map2rdf')
dot_h.edge('plantismash', 'map2rdf')
dot_h.edge('map2rdf', 'g_bgc_plantismash')
dot_h.edge('map2rdf', 'g_bgc_mibig')
dot_h.edge('ncbitaxon_src', 'load_ncbi')
dot_h.edge('load_ncbi', 'g_ncbi')
dot_h.edge('rdf_validate', 'void_create', style='dashed')
dot_h.edge('map2rdf', 'void_create', style='dashed')
dot_h.edge('void_create', 'g_void')
dot_h.edge('g_pathways', 'sparql')
dot_h.edge('g_taxextra', 'sparql')
dot_h.edge('g_propextra', 'sparql')
dot_h.edge('g_bgc_plantismash', 'sparql')
dot_h.edge('g_bgc_mibig', 'sparql')
dot_h.edge('g_ncbi', 'sparql')
dot_h.edge('g_void', 'sparql')
dot_h.edge('sparql', 'snorql')
dot_h.edge('sparql', 'notebooks')
dot_h.edge('snorql', 'sparql_examples', style='dashed', label='GitHub')
dot_h.edge('snorql', 'tutorials', style='dashed')
dot_h.edge('wikidata_src', 'notebooks', style='dashed', label='federated')
dot_h.edge('wikidata_src', 'sparql_examples', style='dashed', label='federated')

# -- Render -----------------------------------------------------------------------
dot_h_path = str(FIG_DIR / 'pipeline_overview_horizontal')
dot_h.render(dot_h_path, cleanup=False)
(FIG_DIR / 'pipeline_overview_horizontal.dot').write_text(dot_h.source, encoding='utf-8')

print(f"Saved SVG: {dot_h_path}.svg")
print(f"Saved DOT: {dot_h_path}.dot")


Saved SVG: figures/output/figures/pipeline_overview_horizontal.svg
Saved DOT: figures/output/figures/pipeline_overview_horizontal.dot


---
## Figure caption / interpretation

**Figure legend -- PlantMetWiki data transformation pipeline (11 stages).**

The diagram shows the four regions of the PlantMetWiki knowledge-graph construction pipeline:

- **Source data** (blue, stage 1): PlantCyc 17.0 biochemical databases (1,162 pathways, 1,316 reactions, 455 ORG-IDs covering 439 NCBI taxa), MIBiG 4.0 (43 BGCs: 9 typed *Arabidopsis thaliana*, 34 untyped/no species assigned) and plantiSMASH v2 (65 predicted BGCs, *Arabidopsis thaliana* only) biosynthetic gene cluster annotations, the OBO Foundry NCBITaxon ontology (CC0), and Wikidata (accessed via federated SPARQL using InChIKey).

- **Pipeline tools** (yellow, stages 2-10): Input validation (stage 2) checks taxonomic consistency before file generation. GPML conversion (stage 3) produces 2,478 GPML2021 files. GPML-to-RDF (stages 4-6) generates three named graphs totalling 6,474,584 triples: core WikiPathways RDF (3,826,567), taxonomy extra (30,178; 424/439 NCBI taxa annotated; 6 taxa absent from OBO Foundry release, affecting 37 DataNodes), and properties extra (2,617,839). RDF validation (stage 7) checks all output TTL files for syntax and content errors (0 errors). The NCBITaxon ontology step (stage 8) uses ROBOT v1.9.6 MIREOT extraction to build a data-driven subset of 424 seed taxa plus full ancestor lineages (17,707 triples, 1.4 MB -- reduced from the 1.8 GB full release). BGC integration (stage 9, `map-to-rdf`) is scoped to *Arabidopsis thaliana* only -- plantiSMASH also predicts clusters for *Solanum lycopersicum*, but those use a different genome annotation that doesn't join to PlantCyc gene IDs (preserved on a separate branch, not part of this release) -- and adds 199 BGC-pathway crosslinks, computed at the SPARQL query layer rather than stored as RDF. VoID metadata creation (stage 10, new in this release) generates machine-readable dataset descriptions -- 5 `void:Dataset` entries (core, taxonomy-extra, properties-extra, plantismash-v2, mibig-4.0) plus 1 `void:Linkset`, 151 triples -- recording each dataset's version, Zenodo deposit DOI, licence, and byte size; see `notebooks/figures/output/named_graphs_metadata.csv` for the full table.

- **Virtuoso triplestore** (green, stage 11): Seven named graphs totalling 6,500,285 triples. The `graph/void` named graph (151 triples) makes the dataset/version/DOI/licence metadata above directly SPARQL-queryable (see `SPARQLQueries/1.(Meta)Data/VoIDHeader.rq`). The `graph/ncbitaxon` named graph provides taxon labels and hierarchical context for all PlantMetWiki taxa -- its own VoID description currently exists only as a static file (`void-ncbitaxon.ttl`) and is not yet loaded into the live `graph/void`.

- **Access and results** (purple): A public SPARQL endpoint exposes all graphs. Snorql-UI provides a browser query interface and loads a curated set of customizable SPARQL query examples from GitHub (github.com/pathway-lod/SPARQLQueries). Tutorial pages document step-by-step SPARQL queries including federated queries linking PlantMetWiki taxa to Wikidata metabolites. Jupyter notebooks support publication figures, cross-species pathway inference, and federated metabolite analyses (4,577 unique metabolites, 4,111 with InChIKey).

All triple counts were independently re-verified directly against the current release bundles (rdflib parse) rather than trusted from a live Virtuoso instance, since a local dev/test instance can carry stale or duplicate graphs from earlier runs. The DOT source files (`pipeline_overview.dot`, `pipeline_overview_horizontal.dot`) are version-controlled alongside this notebook.
